# SIV-WAM 手撕学习 · 03：从 Dataset Batch 到 Training Input

你已经完成了 `02_flow_matching.ipynb`，现在开始训练阶段真正的桥接部分：

```text
Dataset output
    ↓
pad_collate_fn
    ↓
valid_frames
    ↓
video / mask / action 分别加噪
    ↓
latent_dict / mask_dict / action_dict
    ↓
Transformer training interface
```

本 Notebook 会继续留出关键实现，让你自己完成 `pad_collate_fn`、`_add_noise()` 和 `_prepare_input_dict()`。

## 本节核心问题

完成后你必须能解释：

1. 为什么 batch 内不同时间长度必须 padding；
2. 为什么需要 `valid_frames`；
3. video/mask 为什么共享 latent scheduler；
4. action 为什么使用独立 scheduler；
5. mask 为什么复用 video timestep；
6. `latent_dict`、`mask_dict`、`action_dict` 如何对应模型的三个输入流。

In [1]:
from pathlib import Path

import torch
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd()
print('torch:', torch.__version__)

torch: 2.9.0+cu126


## 0. 复用你在 Notebook 02 中完成的 scheduler

请把你在 `02_flow_matching.ipynb` 中已经完成的 scheduler 类复制到下面的 cell，或者将它改造成一个可导入的 Python 类。

本 Notebook 只要求它提供：

```python
scheduler.timesteps
scheduler.add_noise(clean, noise, timestep, t_dim=2)
scheduler.training_target(clean, noise, timestep)
```

为了先聚焦输入构造，下面提供一个临时可运行的 scheduler。你完成自己的版本后，应替换它。

In [2]:
class MinimalScheduler:
    def __init__(self, num_train_timesteps=100, sigma_min=0.0, sigma_max=1.0):
        self.num_train_timesteps = num_train_timesteps
        self.sigmas = torch.linspace(sigma_max, sigma_min, num_train_timesteps)
        self.timesteps = self.sigmas * num_train_timesteps

    def add_noise(self, clean, noise, timestep, t_dim=2):
        timestep = torch.as_tensor(timestep)
        ids = torch.argmin(
            (self.timesteps[:, None] - timestep.reshape(1, -1)).abs(), dim=0
        )
        sigma = self.sigmas[ids].to(clean.device, clean.dtype)
        shape = [1] * clean.ndim
        shape[t_dim] = sigma.numel()
        sigma = sigma.reshape(shape)
        return (1 - sigma) * clean + sigma * noise

    def training_target(self, clean, noise, timestep):
        return noise - clean

latent_scheduler = MinimalScheduler()
action_scheduler = MinimalScheduler()
print(latent_scheduler.timesteps.shape)

torch.Size([100])


## 1. 构造不同时间长度的 fake samples

真实数据中不同窗口的 `F` 可能不同。本节故意构造 `F=2` 和 `F=3` 的两个样本。

In [3]:
def make_sample(F_len, seed):
    g = torch.Generator().manual_seed(seed)
    return {
        'latents': torch.randn(4, F_len, 8, 10, generator=g),
        'mask_latents': torch.randn(4, F_len, 4, 10, generator=g),
        'actions': torch.randn(30, F_len, 16, 1, generator=g),
        'actions_mask': torch.ones(30, F_len, 16, 1, dtype=torch.bool),
        'text_emb': torch.randn(8, 16, generator=g),
    }

samples = [make_sample(2, 0), make_sample(3, 1)]
for i, sample in enumerate(samples):
    print(i, {k: tuple(v.shape) for k, v in sample.items()})

0 {'latents': (4, 2, 8, 10), 'mask_latents': (4, 2, 4, 10), 'actions': (30, 2, 16, 1), 'actions_mask': (30, 2, 16, 1), 'text_emb': (8, 16)}
1 {'latents': (4, 3, 8, 10), 'mask_latents': (4, 3, 4, 10), 'actions': (30, 3, 16, 1), 'actions_mask': (30, 3, 16, 1), 'text_emb': (8, 16)}


## 2. 手写 `pad_collate_fn()`

生产代码会：

- 找到 batch 内最大的 latent frame 数；
- 沿时间维补零；
- 同样 padding `latents`、`mask_latents`、`actions`、`actions_mask`；
- 生成 `[B,F]` 的 `valid_frames`。

In [ ]:

def pad_dim1_todo(tensor, target_length, value=0):
    # TODO：只沿 tensor 的第 1 维补到 target_length
    # 提示：tensor 的 shape 是 [C,F,...]，不要改变 C 或空间维
    current_length = tensor.shape[1] # F
    diff = target_length - current_length
    if diff < 0:
        raise ValueError(f"current length {current_length} < target_length {target_length}")
    elif diff == 0:
        return tensor
    else:
        padding = [0] * (2 * (tensor.ndim - 2)) + [0, diff] # 总维度 4， 第一维度之后才需要补0，所以减 2，然后每个维度是一对所以 *2
    return F.pad(tensor,padding,mode="constant",value=value) # 从左开始补W F.pad() 的参数是从最后一维开始解释：
    #  (w_left, w_right, h_left, h_right, f_left, f_right)


def pad_collate_fn_todo(batch):
    # TODO 1：找到 max_lat_f 和 max_act_f
    max_lat_f = max(sample["latents"].shape[1] for sample in batch)
    max_act_f = max(sample["actions"].shape[1] for sample in batch)
    original_lat_f = [sample["latents"].shape[1] for sample in batch]
    # TODO 2：对每个字段调用 pad_dim1_todo
    out = {}
    for key in batch[0]:
        values = []
        for sample in batch:
            value = sample[key]
            if key in ("latents", "mask_latents"):
                value = pad_dim1_todo(value, max_lat_f)
            elif key in ("actions","actions_mask"):
                value = pad_dim1_todo(value, max_act_f)

            values.append(value)
    # TODO 3：torch.stack 成 batch
        out[key] = torch.stack(values)
    # TODO 4：生成 valid_frames
    valid_frames = []
    for real_length in original_lat_f:
        frame_mask = torch.zeros(max_lat_f, dtype=torch.bool)
        frame_mask[:real_length] = True
        valid_frames.append(frame_mask)
    out["valid_frames"] = torch.stack(valid_frames)
    return out

{'latents': tensor([[[[[-1.1258e+00, -1.1524e+00, -2.5058e-01,  ..., -2.1152e+00,
              3.2227e-01, -1.2633e+00],
            [ 3.4998e-01,  3.0813e-01,  1.1984e-01,  ..., -1.6959e+00,
              5.6665e-01,  7.9351e-01],
            [ 5.9884e-01, -1.5551e+00, -3.4136e-01,  ...,  1.8348e-01,
              1.3894e+00,  1.5863e+00],
            ...,
            [ 1.0554e+00,  1.7784e-01, -2.3034e-01,  ...,  7.4402e-01,
              1.5210e+00,  3.4105e+00],
            [-1.5312e+00, -1.2341e+00,  1.8197e+00,  ...,  1.2899e+00,
             -1.4782e+00,  2.5672e+00],
            [-4.7312e-01,  3.3555e-01, -1.6293e+00,  ...,  1.1149e+00,
             -1.4067e-01,  8.0575e-01]],
 
           [[-9.3348e-02,  6.8705e-01, -8.3832e-01,  ...,  3.5815e-01,
             -2.4600e-01,  2.3025e+00],
            [-1.8817e+00, -4.9727e-02, -1.0450e+00,  ..., -1.3602e+00,
              3.4457e-01,  5.1987e-01],
            [-2.6133e+00, -1.6965e+00, -2.2824e-01,  ...,  4.5440e-01,
          

In [56]:
# 完成上一个 cell 后运行
batch = pad_collate_fn_todo(samples)
for key, value in batch.items():
    print(key, tuple(value.shape), value.dtype)

assert batch['latents'].shape == (2, 4, 3, 8, 10)
assert batch['mask_latents'].shape == (2, 4, 3, 4, 10)
assert batch['actions'].shape == (2, 30, 3, 16, 1)
assert batch['actions_mask'].dtype == torch.bool
assert batch['valid_frames'].tolist() == [[True, True, False], [True, True, True]]

latents (2, 4, 3, 8, 10) torch.float32
mask_latents (2, 4, 3, 4, 10) torch.float32
actions (2, 30, 3, 16, 1) torch.float32
actions_mask (2, 30, 3, 16, 1) torch.bool
text_emb (2, 8, 16) torch.float32
valid_frames (2, 3) torch.bool


## 3. 手写 `_add_noise()`

当前先实现核心版：

```text
输入 clean latent [B,C,F,H,W]
    ↓ 为每个 F 采样 timestep
    ↓ 生成 noise
    ↓ scheduler.add_noise
    ↓ scheduler.training_target
输出 noisy_latents、targets、timesteps、latent
```

注意：生产代码还会处理 `cond_timesteps`、`grid_id` 和 action mask；先完成最核心的数据流。

In [57]:
def sample_timestep_id_simple(num_samples, num_train_timesteps):
    # TODO：可以复用 Notebook 02 中你已经写好的实现
    u = torch.rand(num_samples)
    return (u * num_train_timesteps).clamp(0, num_train_timesteps - 1).long()


def add_noise_todo(latent, scheduler, action_mask=None, action_mode=False):
    # TODO 1：读取 B,C,F,H,W
    B,C,F,H,W = latent.shape
    # TODO 2：为 F 个时间帧采样 timestep ids
    timestep_ids = sample_timestep_id_simple(F, scheduler.num_train_timesteps)
    # TODO 3：生成与 latent 同 shape 的 noise, 随机高斯噪声
    noise = torch.randn_like(latent)
    # TODO 4：调用 scheduler.add_noise 和 training_target
    timesteps = scheduler.timesteps[timestep_ids].to(latent.device)
    latent_noise = scheduler.add_noise(latent, noise, timesteps)
    target = scheduler.training_target(latent, noise, timesteps)
    # TODO 5：如果 action_mask 非空，对 noisy/target/latent 都应用 mask
    if action_mask is not None:
        action_mask = action_mask.to(device=latent.device, dtype=torch.bool)
        latent_noise *= action_mask
        target *= action_mask
        latent *= action_mask
    timesteps_batch = timesteps[None].repeat(B,1)
    return {
        "timesteps": timesteps_batch,
        "noisy_latents": latent_noise,
        "targets": target,
        "latent": latent,
        "noise": noise
    }

In [58]:
# 完成 add_noise_todo 后运行
video_dict = add_noise_todo(batch['latents'], latent_scheduler)
action_dict = add_noise_todo(
    batch['actions'], action_scheduler, batch['actions_mask'], action_mode=True
)

for name, item in [('video', video_dict), ('action', action_dict)]:
    print(name)
    for k, v in item.items():
        if torch.is_tensor(v):
            print(' ', k, tuple(v.shape), v.dtype)

assert video_dict['noisy_latents'].shape == batch['latents'].shape
assert action_dict['noisy_latents'].shape == batch['actions'].shape
assert video_dict['timesteps'].shape == (2, 3)
assert action_dict['timesteps'].shape == (2, 3)

video
  timesteps (2, 3) torch.float32
  noisy_latents (2, 4, 3, 8, 10) torch.float32
  targets (2, 4, 3, 8, 10) torch.float32
  latent (2, 4, 3, 8, 10) torch.float32
  noise (2, 4, 3, 8, 10) torch.float32
action
  timesteps (2, 3) torch.float32
  noisy_latents (2, 30, 3, 16, 1) torch.float32
  targets (2, 30, 3, 16, 1) torch.float32
  latent (2, 30, 3, 16, 1) torch.float32
  noise (2, 30, 3, 16, 1) torch.float32


## 4. 手写 `_prepare_input_dict()`

现在把三个分支组合起来：

```text
video  → latent_dict
mask   → mask_dict
action → action_dict
```

核心约束：

- mask 使用 latent scheduler；
- mask noisy latent 复用 video timestep；
- action 使用 action scheduler；
- mask target 与 video 使用相同的时间帧数；
- `mask_dict['latent']` 保留 clean mask condition。

先完成一个不含 `grid_id` 的版本；`grid_id` 会在下一阶段 token layout 中补上。

In [60]:
def prepare_input_dict_todo(
    batch,
    latent_scheduler,
    action_scheduler,
):
    # 1. 视频：使用 latent scheduler，独立采样时间步并加噪
    latent_dict = add_noise_todo(
        latent=batch["latents"],
        scheduler=latent_scheduler,
    )

    # 2. 动作：使用 action scheduler，应用动作通道 mask
    action_dict = add_noise_todo(
        latent=batch["actions"],
        scheduler=action_scheduler,
        action_mask=batch["actions_mask"],
        action_mode=True,
    )

    # 3. 文本和动作 mask 也是模型需要的条件
    latent_dict["text_emb"] = batch["text_emb"]
    action_dict["text_emb"] = batch["text_emb"]
    action_dict["actions_mask"] = batch["actions_mask"]

    # 4. mask 使用与视频相同的 latent scheduler 和时间步
    clean_mask = batch["mask_latents"]
    video_timesteps = latent_dict["timesteps"]  # [B, F]

    if clean_mask.shape[2] != video_timesteps.shape[1]:
        raise ValueError(
            "mask 的时间帧数必须与视频 timesteps 的 F 一致"
        )

    mask_noise = torch.randn_like(clean_mask)

    # 当前 Notebook 的 scheduler 接收 [F]，而不是 [B,F]。
    # 视频时间步在 add_noise_todo 中被复制给整个 batch，
    # 因此取第一个样本即可。
    timesteps_f = video_timesteps[0]

    noisy_mask = latent_scheduler.add_noise(
        clean_mask,
        mask_noise,
        timesteps_f,
        t_dim=2,
    )

    mask_targets = latent_scheduler.training_target(
        clean_mask,
        mask_noise,
        timesteps_f,
    )

    mask_dict = {
        "noisy_latents": noisy_mask,
        "latent": clean_mask,          # 干净的 mask 条件
        "timesteps": video_timesteps,  # 与视频完全相同：[B,F]
        "targets": mask_targets,
    }

    return {
        "latent_dict": latent_dict,
        "mask_dict": mask_dict,
        "action_dict": action_dict,
        "mask_targets": mask_targets,
    }

In [61]:
# 完成 prepare_input_dict_todo 后运行
input_dict = prepare_input_dict_todo(batch, latent_scheduler, action_scheduler)
print(input_dict.keys())
for stream in ['latent_dict', 'mask_dict', 'action_dict']:
    print('\n', stream)
    for key, value in input_dict[stream].items():
        if torch.is_tensor(value):
            print(key, tuple(value.shape))

assert torch.equal(
    input_dict["mask_dict"]["timesteps"],
    input_dict["latent_dict"]["timesteps"],
)

assert input_dict["mask_dict"]["noisy_latents"].shape == batch["mask_latents"].shape
assert input_dict["mask_targets"].shape == batch["mask_latents"].shape
assert input_dict["action_dict"]["noisy_latents"].shape == batch["actions"].shape

for stream in ("latent_dict", "mask_dict", "action_dict"):
    print(stream)
    for name, value in input_dict[stream].items():
        if torch.is_tensor(value):
            print(f"  {name:16s} {tuple(value.shape)}")

dict_keys(['latent_dict', 'mask_dict', 'action_dict', 'mask_targets'])

 latent_dict
timesteps (2, 3)
noisy_latents (2, 4, 3, 8, 10)
targets (2, 4, 3, 8, 10)
latent (2, 4, 3, 8, 10)
noise (2, 4, 3, 8, 10)
text_emb (2, 8, 16)

 mask_dict
noisy_latents (2, 4, 3, 4, 10)
latent (2, 4, 3, 4, 10)
timesteps (2, 3)
targets (2, 4, 3, 4, 10)

 action_dict
timesteps (2, 3)
noisy_latents (2, 30, 3, 16, 1)
targets (2, 30, 3, 16, 1)
latent (2, 30, 3, 16, 1)
noise (2, 30, 3, 16, 1)
text_emb (2, 8, 16)
actions_mask (2, 30, 3, 16, 1)
latent_dict
  timesteps        (2, 3)
  noisy_latents    (2, 4, 3, 8, 10)
  targets          (2, 4, 3, 8, 10)
  latent           (2, 4, 3, 8, 10)
  noise            (2, 4, 3, 8, 10)
  text_emb         (2, 8, 16)
mask_dict
  noisy_latents    (2, 4, 3, 4, 10)
  latent           (2, 4, 3, 4, 10)
  timesteps        (2, 3)
  targets          (2, 4, 3, 4, 10)
action_dict
  timesteps        (2, 3)
  noisy_latents    (2, 30, 3, 16, 1)
  targets          (2, 30, 3, 16, 1)
  latent 

## 本节验收标准

完成后你应该能画出：

```text
batch [B,...]
   ↓
pad_collate_fn
   ├── valid_frames [B,F]
   ├── video + latent scheduler → latent_dict
   ├── mask + video timesteps → mask_dict
   └── action + action scheduler → action_dict
```

请先完成以下三个 TODO：

1. `pad_dim1_todo` / `pad_collate_fn_todo`；
2. `add_noise_todo`；
3. `prepare_input_dict_todo`。

完成后下一阶段是 `04_token_layout.ipynb`：把这三个 dict 变成六类 token，并构造 Transformer 的输入序列。